In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
#from sklearn.manifold import TSNE
from scipy import integrate as int
from scipy import stats
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn import datasets
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
import statsmodels.stats.weightstats as ws
from sklearn.cluster import KMeans
import umap
from lmfit import minimize, Parameters
import matplotlib 
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler


import matplotlib.pyplot as pl
from sklearn.cluster import AgglomerativeClustering
hKWD={'stat':'density','element':'step','fill':False}

# Define a bunch of functions

In [ ]:

from scipy.stats import norm

def qqFit(data1,data2,data3):
    data1_quantiles = np.percentile(data1, np.linspace(0, 100, 50))#len(data3)))
    data2_quantiles = np.percentile(data2, np.linspace(0, 100, 50))#len(data3)))
    # we assume the distribution of data1,data2 is the same  
    # changes are due to machne error (they are anchor to each other) 
    # create lookup table where for any quantile q2_i in data2  
    # f(q2_i)=q1_i:
    # data2_into_data1 =np.interp(data3, data2_quantiles, data1_quantiles)
    
    # data3 is not with the same distribution as data1,data2 but with same errors as data 2
    # i.e we transfer it to be with same errors as data1 to be able to compare to each other
    data3_into_data1 = np.interp(data3, data2_quantiles, data1_quantiles)
    return data3_into_data1

def LinearFit(data1,data2,data3):
    # we assume the distribution of data1,data2 is the same  
    # changes are due to machine error (they are anchor to each other) 
    # create lookup table where for any quantile q2_i in data2  

    
    # data3 is not with the same distribution as data1,data2 but with same errors as data 2
    # i.e we transfer it to be with same errors as data1 to be able to compare to each other
    mu1,sigma1 = norm.fit(data1)
    mu2,sigma2 = norm.fit(data2)

    # mu1 = np.mean(data1)
    # sigma1 = np.std(data1)
    # mu2 = np.mean(data2)
    # sigma2 = np.std(data2)

    data3_into_data1 = (data3/sigma2-mu2/sigma2 + mu1/sigma1)*sigma1
    return data3_into_data1

def MixedFit(data1,data2,data3):

    # we assume the distribution of data1,data2 is the same  
    # changes are due to machine error (they are anchor to each other) 
    # create lookup table where for any quantile q2_i in data2  
    # f(q2_i)=q1_i:
    # data2_into_data1 =np.interp(data3, data2_quantiles, data1_quantiles)
    
    # data3 is not with the same distribution as data1,data2 but with same errors as data 2
    # i.e we transfer it to be with same errors as data1 to be able to compare to each other
    data1 = np.array(data1).copy()
    data2 = np.array(data2).copy()
    data3 = np.array(data3).copy()

    # data2_quantiles = np.percentile(data2, np.linspace(0, 100, len(data2)))
    # ind = (data3>np.max(data2_quantiles))*1 +(data3<np.min(data2_quantiles))*1
    ind = (data3>np.max(data2))*1 +(data3<np.min(data2))*1
    ind1 =[i for i in np.arange(data3.shape[0]) if ind[i]==1]
    ind2 =[i for i in np.arange(data3.shape[0]) if ind[i]==0]
    
    data3_into_data1 = np.zeros(data3.shape[0])

    data3_into_data1[ind1] = LinearFit(data1,data2,data3[ind1])
    data3_into_data1[ind2] = qqFit(data1,data2,data3[ind2])
    

    return data3_into_data1 

def fitDF(fit_into,fit_from,df_to_fit,method):
    cols = [col for col in fit_into.columns if col in fit_from.columns]
    dropped_cols = [col for col in df_to_fit.columns if col not in cols]
    if len(dropped_cols)>0:
        print (f'cols dropped due to fitting: {dropped_cols} ')

    fitted_df = pd.DataFrame(columns=cols)
    for col in cols:
        print(col)
        fitted_df[col] = method(fit_into[col],fit_from[col],df_to_fit[col])
    return fitted_df,dropped_cols

def TestHetero(DB1,DB2,UMAPMRK,CNum=20000,fname=None):
        from scipy.spatial import distance
        CAll=pd.concat([
                DB1.sample(CNum,random_state=42),
                DB2.sample(CNum,random_state=42)
        ]).copy()
        X_2d=draw_umap(CAll[UMAPMRK],cc=CAll['H4'],min_dist=0.001,n_neighbors=60,rstate=42)
        L1=DB1.iloc[0].Line

        L2=DB2.iloc[0].Line
        
        m=CAll['Line']==DB1.loc[0].Line
        plt.scatter(X_2d[m,0],X_2d[m,1],c='b',s=1,label=L1);
        plt.scatter(X_2d[~m,0],X_2d[~m,1],c='r',s=1,label=L2);
        plt.legend(markerscale=10)
        if fname is not None:
            plt.savefig(fname)
            
        xmax=X_2d[:,0].max()
        xmin=X_2d[:,0].min()
        ymax=X_2d[:,0].max()
        ymin=X_2d[:,0].min()
        mx=np.max([xmax,ymax])
        mn=np.min([xmin,ymin])
        
        m=CAll.Line==L1
        b=np.linspace(round(mn,0)-1,round(mx,0)+1,50)
        A,_,_=np.histogram2d(X_2d[m,0],X_2d[m,1],bins=b)
        # plt.figure(figsize=(5,5))
        # plt.imshow(A>0)
        DD=distance.cdist(X_2d[m],X_2d[m]).flatten()
        print(L1," Local: ",(A>0).sum()," Global: ",np.round(np.quantile(DD,0.95),2))

        m=CAll.Line==L2
        b=np.linspace(round(mn,0)-1,round(mx,0)+1,50)
        A,_,_=np.histogram2d(X_2d[m,0],X_2d[m,1],bins=b)
        # plt.figure(figsize=(5,5))
        # plt.imshow(A>0)
        DD=distance.cdist(X_2d[m],X_2d[m]).flatten()
        print(L2," Local: ", (A>0).sum()," Global: ",np.round(np.quantile(DD,0.95),2))
              
            
def wfall(shap_values, max_display=10, show=True):
    """ Plots an explantion of a single prediction as a waterfall plot.
    The SHAP value of a feature represents the impact of the evidence provided by that feature on the model's
    output. The waterfall plot is designed to visually display how the SHAP values (evidence) of each feature
    move the model output from our prior expectation under the background data distribution, to the final model
    prediction given the evidence of all the features. Features are sorted by the magnitude of their SHAP values
    with the smallest magnitude features grouped together at the bottom of the plot when the number of features
    in the models exceeds the max_display parameter.
    
    Parameters
    ----------
    shap_values : Explanation
        A one-dimensional Explanation object that contains the feature values and SHAP values to plot.
    max_display : str
        The maximum number of features to plot.
    show : bool
        Whether matplotlib.pyplot.show() is called before returning. Setting this to False allows the plot
        to be customized further after it has been created.
    """
    dark_o= mpl.colors.to_rgb('dimgray')
    dim_g= mpl.colors.to_rgb('darkorange')

    base_values = shap_values.base_values
    
    features = shap_values.data
    feature_names = shap_values.feature_names
    lower_bounds = getattr(shap_values, "lower_bounds", None)
    upper_bounds = getattr(shap_values, "upper_bounds", None)
    values = shap_values.values

    # make sure we only have a single output to explain
    if (type(base_values) == np.ndarray and len(base_values) > 0) or type(base_values) == list:
        raise Exception("waterfall_plot requires a scalar base_values of the model output as the first " \
                        "parameter, but you have passed an array as the first parameter! " \
                        "Try shap.waterfall_plot(explainer.base_values[0], values[0], X[0]) or " \
                        "for multi-output models try " \
                        "shap.waterfall_plot(explainer.base_values[0], values[0][0], X[0]).")

    # make sure we only have a single explanation to plot
    if len(values.shape) == 2:
        raise Exception("The waterfall_plot can currently only plot a single explanation but a matrix of explanations was passed!")
    
    # unwrap pandas series
    if safe_isinstance(features, "pandas.core.series.Series"):
        if feature_names is None:
            feature_names = list(features.index)
        features = features.values

    # fallback feature names
    if feature_names is None:
        feature_names = np.array([labels['FEATURE'] % str(i) for i in range(len(values))])
    
    # init variables we use for tracking the plot locations
    num_features = min(max_display, len(values))
    row_height = 0.5
    rng = range(num_features - 1, -1, -1)
    order = np.argsort(-np.abs(values))
    pos_lefts = []
    pos_inds = []
    pos_widths = []
    pos_low = []
    pos_high = []
    neg_lefts = []
    neg_inds = []
    neg_widths = []
    neg_low = []
    neg_high = []
    loc = base_values + values.sum()
    yticklabels = ["" for i in range(num_features + 1)]
    
    # size the plot based on how many features we are plotting
    pl.gcf().set_size_inches(8, num_features * row_height + 1.5)

    # see how many individual (vs. grouped at the end) features we are plotting
    if num_features == len(values):
        num_individual = num_features
    else:
        num_individual = num_features - 1

    # compute the locations of the individual features and plot the dashed connecting lines
    for i in range(num_individual):
        sval = values[order[i]]
        loc -= sval
        if sval >= 0:
            pos_inds.append(rng[i])
            pos_widths.append(sval)
            if lower_bounds is not None:
                pos_low.append(lower_bounds[order[i]])
                pos_high.append(upper_bounds[order[i]])
            pos_lefts.append(loc)
        else:
            neg_inds.append(rng[i])
            neg_widths.append(sval)
            if lower_bounds is not None:
                neg_low.append(lower_bounds[order[i]])
                neg_high.append(upper_bounds[order[i]])
            neg_lefts.append(loc)
        if num_individual != num_features or i + 4 < num_individual:
            pl.plot([loc, loc], [rng[i] -1 - 0.4, rng[i] + 0.4], color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
        if features is None:
            yticklabels[rng[i]] = feature_names[order[i]]
        else:
            yticklabels[rng[i]] = format_value(features[order[i]], "%0.03f") + " = " + feature_names[order[i]] 
    
    # add a last grouped feature to represent the impact of all the features we didn't show
    if num_features < len(values):
        yticklabels[0] = "%d other features" % (len(values) - num_features + 1)
        remaining_impact = base_values - loc
        if remaining_impact < 0:
            pos_inds.append(0)
            pos_widths.append(-remaining_impact)
            pos_lefts.append(loc + remaining_impact)
            c = dim_g  #colors.red_rgb
        else:
            neg_inds.append(0)
            neg_widths.append(-remaining_impact)
            neg_lefts.append(loc + remaining_impact)
            c = dark_o #colors.blue_rgb

    points = pos_lefts + list(np.array(pos_lefts) + np.array(pos_widths)) + neg_lefts + list(np.array(neg_lefts) + np.array(neg_widths))
    dataw = np.max(points) - np.min(points)
    
    # draw invisible bars just for sizing the axes
    label_padding = np.array([0.1*dataw if w < 1 else 0 for w in pos_widths])
    pl.barh(pos_inds, np.array(pos_widths) + label_padding + 0.02*dataw, left=np.array(pos_lefts) - 0.01*dataw, color=colors.red_rgb, alpha=0)
    label_padding = np.array([-0.1*dataw  if -w < 1 else 0 for w in neg_widths])
    pl.barh(neg_inds, np.array(neg_widths) + label_padding - 0.02*dataw, left=np.array(neg_lefts) + 0.01*dataw, color=colors.blue_rgb, alpha=0)
    
    # define variable we need for plotting the arrows
    head_length = 0.08
    bar_width = 0.8
    xlen = pl.xlim()[1] - pl.xlim()[0]
    fig = pl.gcf()
    ax = pl.gca()
    xticks = ax.get_xticks()
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    width, height = bbox.width, bbox.height
    bbox_to_xscale = xlen/width
    hl_scaled = bbox_to_xscale * head_length
    renderer = fig.canvas.get_renderer()
    
    # draw the positive arrows
    for i in range(len(pos_inds)):
        dist = pos_widths[i]
        arrow_obj = pl.arrow(
            pos_lefts[i], pos_inds[i], max(dist-hl_scaled, 0.000001), 0,
            head_length=min(dist, hl_scaled),
            color=dim_g, width=bar_width,
            head_width=bar_width
        )
        
        if pos_low is not None and i < len(pos_low):
            pl.errorbar(
                pos_lefts[i] + pos_widths[i], pos_inds[i], 
                xerr=np.array([[pos_widths[i] - pos_low[i]], [pos_high[i] - pos_widths[i]]]),
                ecolor=dim_g
            )

        txt_obj = pl.text(
            pos_lefts[i] + 0.5*dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                pos_lefts[i] + (5/72)*bbox_to_xscale + dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
                horizontalalignment='left', verticalalignment='center', color=dim_g,
                fontsize=12
            )
    
    # draw the negative arrows
    for i in range(len(neg_inds)):
        dist = neg_widths[i]
        
        arrow_obj = pl.arrow(
            neg_lefts[i], neg_inds[i], -max(-dist-hl_scaled, 0.000001), 0,
            head_length=min(-dist, hl_scaled),
            color=dark_o, width=bar_width,
            head_width=bar_width
        )

        if neg_low is not None and i < len(neg_low):
            pl.errorbar(
                neg_lefts[i] + neg_widths[i], neg_inds[i], 
                xerr=np.array([[neg_widths[i] - neg_low[i]], [neg_high[i] - neg_widths[i]]]),
                ecolor=dark_o
            )
        
        txt_obj = pl.text(
            neg_lefts[i] + 0.5*dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                neg_lefts[i] - (5/72)*bbox_to_xscale + dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
                horizontalalignment='right', verticalalignment='center', color=dark_o,
                fontsize=12
            )

    # draw the y-ticks twice, once in gray and then again with just the feature names in black
    ytick_pos = list(range(num_features)) + list(np.arange(num_features)+1e-8) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    pl.yticks(ytick_pos, yticklabels[:-1] + [l.split('=')[-1] for l in yticklabels[:-1]], fontsize=13)
    
    # put horizontal lines for each feature row
    for i in range(num_features):
        pl.axhline(i, color="#cccccc", lw=0.5, dashes=(1, 5), zorder=-1)
    
    # mark the prior expected value and the model prediction
    pl.axvline(base_values, 0, 1/num_features, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    fx = base_values + values.sum()
    pl.axvline(fx, 0, 1, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    
    # clean up the main axis
    pl.gca().xaxis.set_ticks_position('bottom')
    pl.gca().yaxis.set_ticks_position('none')
    pl.gca().spines['right'].set_visible(False)
    pl.gca().spines['top'].set_visible(False)
    pl.gca().spines['left'].set_visible(False)
    ax.tick_params(labelsize=13)
    #pl.xlabel("\nModel output", fontsize=12)

    # draw the E[f(X)] tick mark
    xmin,xmax = ax.get_xlim()
    ax2=ax.twiny()
    ax2.set_xlim(xmin,xmax)
    ax2.set_xticks([base_values, base_values+1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax2.set_xticklabels(["\n$E[f(X)]$","\n$ = "+format_value(base_values, "%0.03f")+"$"], fontsize=12, ha="left")
    ax2.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)

    # draw the f(x) tick mark
    ax3=ax2.twiny()
    ax3.set_xlim(xmin,xmax)
    ax3.set_xticks([base_values + values.sum(), base_values + values.sum() + 1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax3.set_xticklabels(["$f(x)$","$ = "+format_value(fx, "%0.03f")+"$"], fontsize=12, ha="left")
    tick_labels = ax3.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-10/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(12/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_color("#999999")
    ax3.spines['right'].set_visible(False)
    ax3.spines['top'].set_visible(False)
    ax3.spines['left'].set_visible(False)

    # adjust the position of the E[f(X)] = x.xx label
    tick_labels = ax2.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-20/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(22/72., -1/72., fig.dpi_scale_trans))
    
    tick_labels[1].set_color("#999999")

    # color the y tick labels that have the feature values as gray
    # (these fall behind the black ones with just the feature name)
    tick_labels = ax.yaxis.get_majorticklabels()
    for i in range(num_features):
        tick_labels[i].set_color("#999999")
    
    if show:
        pl.show()

def dbscan_plot(data,eps=0.1,min_samples=50):
    X=data
    X = StandardScaler().fit_transform(X)
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
    core_samples_mask[db.core_sample_indices_] = True
    labels = db.labels_

    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    print('Estimated number of clusters: %d' % n_clusters_)
    print('Estimated number of noise points: %d' % n_noise_)
    print("Silhouette Coefficient: %0.3f"
          % metrics.silhouette_score(X, labels))

    # Black removed and is used for noise instead.
    plt.figure(figsize=(10, 10))
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each)
              for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]

        class_member_mask = (labels == k)
        
        xy = X[class_member_mask & core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),label = k,
                 markeredgecolor='k', markersize=14)
        
        xy = X[class_member_mask & ~core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                 markeredgecolor='k', markersize=6)
    
    plt.legend(fontsize=15, title_fontsize='40')    
    plt.title('Estimated number of clusters: %d' % n_clusters_)
#    plt.show()
    return labels



def residual(params, x, data):
    alpha = params['alpha']
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H3.3']*alpha+x['H4']*beta+x['H3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H3.3'])+np.std(od['H4'])+np.std(od['H3'])


def residual2(params, x, data):
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H4']*beta+x['H3.3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H4'])+np.std(od['H3.3'])



def twoSampZ(X1, X2):
    from numpy import sqrt, abs, round
    from scipy.stats import norm
    mudiff=np.mean(X1)-np.mean(X2)
    sd1=np.std(X1)
    sd2=np.std(X2)
    n1=len(X1)
    n2=len(X2)
    pooledSE = sqrt(sd1**2/n1 + sd2**2/n2)
    z = ((X1 - X2) - mudiff)/pooledSE
    pval = 2*(1 - norm.cdf(abs(z)))
    return round(pval, 4)

def statistic(dframe):
    return dframe.corr().loc[Var1,Var2]


def draw_umap(data,n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean', title=''
              ,cc=0,rstate=42,dens=False):
    fit = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric, random_state=rstate, verbose=True, densmap=dens
    )
    u = fit.fit_transform(data);
    plt.figure(figsize=(6, 5))
    if n_components == 2:
        plt.scatter(u[:,0], u[:,1], c=cc,s=3,cmap=plt.cm.seismic)
        plt.clim(-5,5)
        plt.colorbar()
    plt.title(title, fontsize=18)
    return u;


def NormMark(data):
    params = Parameters()
    params.add('beta', value=0.1, min=0)
    params.add('gamma', value=0.1, min=0)
    params.add('alpha', value=0.1, min=0)
    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value
    alpha=out.params['alpha'].value
    avMarkers=ddf['H3.3']*alpha+ddf['H4']*beta+ddf['H3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data

def NormMark2(data):
    params = Parameters()
    params.add('beta', value=0.1, min=-1000)
    params.add('gamma', value=0.1, min=-1000)

    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual2, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value

    avMarkers=ddf['H4']*beta+ddf['H3.3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols_M]=data[EpiCols_M]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data






def f(): raise Exception("Found exit()")



def BPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.boxplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   

def VPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.violinplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   


def KPlots(data,NMS,titleSup=''):
    for NN in NMS:
        plt.figure(figsize=(10,10))
        sns.kdeplot(data=data,x=NN,color='blue')
        
#        plt.legend()
        plt.title(""+NN+" "+titleSup)
        plt.show()



def MeanDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

    
def MedDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].median().sort_values(ascending=False)
    dd1=data2[Markers].median().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)    
    
def MeanDistIdU(data1,data2,Markers,title=''):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    
    colors = ['dodgerblue' if x < 0 else 'darkmagenta' for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

def KPlot_Mrk(Mark,titleSup=''):
    plt.figure(figsize=(10,10))
    sns.kdeplot(data=C01,x=Mark,label="C01")
    sns.kdeplot(data=C02,x=Mark,label="C02")
    sns.kdeplot(data=C03,x=Mark,label="C03")
    sns.kdeplot(data=C04,x=Mark,label="C04")
    sns.kdeplot(data=C05,x=Mark,label="C05")
    plt.legend()
    plt.title(""+Mark+" "+titleSup)
    plt.show()
    
    
def MeanDistReSamp(data1,data2,Markers,title='',clr=['darkgreen','purple'],nsamp=10,f=0.5):
    sns.set_style({'legend.frameon':True})
    diffs=[]
    for i in range(nsamp):  
        D1=data1.sample(frac=f).copy()
        D2=data2.sample(frac=f).copy()
        dd0=D1[Markers].mean()#.sort_values(ascending=False)
        dd1=D2[Markers].mean()#.sort_values()
        diff=(dd1-dd0)#.sort_values(ascending=False)    
        diffs.append(diff)

    Mdiff=np.asarray(diffs)
    D=pd.DataFrame({'M':Mdiff.mean(axis=0),'S':Mdiff.std(axis=0)},index=Markers)    
    
    diffs=D.sort_values(by='M',ascending=False).copy()
    
    
    colors = [clr[0] if x < 0 else clr[1] for x in diffs.M]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs.M, color=colors, alpha=1, linewidth=5)
    plt.errorbar(y=diffs.index,x=diffs.M,xerr=diffs.S,capsize=5,fmt='k.')
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)            
    

def UMAP_Plot(data1,data2,Markers,Set1='C01',Set2='Other',titleSup=''):
    data1=data1.assign(Set=Set1)
    data2=data2.assign(Set=Set2)
    CAll=data1.append(data2).sample(frac=0.1).copy()
    print(CAll)
    X_2d=draw_umap(CAll[Markers],cc=CAll['H3'],min_dist=0.01)
    for NN in NamesAll:
        cc=CAll[NN]#[mask]
        plt.figure(figsize=(6, 5))
        plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                    c=cc, cmap=plt.cm.jet)
    #    cmap = matplotlib.cm.get_cmap('jet')
        plt.colorbar()
    #    plt.clim(-3.5,3.5)
        plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    #    mask=CAllmask[TSNEVar]==True
    #    rgba = cmap(-10)
    #    plt.scatter(X_2d[mask][:,0],X_2d[mask][:,1],s=2,
    #                color=rgba) 
        plt.title(NN+" "+titleSup)
        plt.show()

    plt.figure(figsize=(6, 5))
    mask=CAll.Set==Set1
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='blue', label=Set1)        
    mask=CAll.Set==Set2
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='red', label=Set2)        
    plt.legend()
    plt.show()
       

def DeltaCorr(data1,data2,Markers,titleSup=''):
    params = {'axes.titlesize': 30,
              'legend.fontsize': 20,
              'figure.figsize': (16, 10),
              'axes.labelsize': 20,
              'axes.titlesize': 20,
              'xtick.labelsize': 16,
              'ytick.labelsize': 16,
              'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

    print(titleSup)
    plt.figure(figsize=(20,20))
    matrix=data2[Markers].corr()-data1[Markers].corr()
    g=sns.clustermap(matrix, annot=True, annot_kws={"size":8},
                     cmap=plt.cm.jet,vmin=matrix.min().min(),vmax=matrix.max().max(),linewidths=.1); 
    plt.xticks(rotation=0); 
    plt.yticks(rotation=0); 

    plt.title(titleSup)
    plt.show()
    
    
def DefStyle():
    params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
    plt.rcParams.update(params)
#    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

# Load and initialize

In [ ]:
NamesAll=['H3',
'Cytokeratin5',
'H3K27me2',
'p53',
'EZH2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'ZEB1',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'human-EpCAM',
'yH2A.X',
'Vimentin',
'ER',
'CD49f',
'CD24',
'GATA3',
'CD44',
'Ki-67',
'K8_18']


IdentityCols=[
'Cytokeratin5',
'GATA3',
'human-EpCAM',
'Vimentin',
'ER',
'CD49f',
'CD24',
'CD44',
'K8_18',
'ZEB1']

NormMRK = ['H3',
'Ki-67',
'p53',
'EZH2',
'Cytokeratin5',
'GATA3',
'Vimentin',
'ER',
'K8_18',
'ZEB1',
'H3K27me2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'yH2A.X',
]


EpiCols=['H3',
'H3K27me2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'yH2A.X',
]

metals_names_map = {' In115Di':'H3', ' Ce140Di':'Cytokeratin5', ' Nd142Di':'H3K27me2', ' Nd143Di':'p53', ' Nd144Di':'EZH2', ' Nd145Di':'H3K4me3',
                   ' Sm149Di':'H3K36me2', ' Nd150Di':'H3K4me1', ' Eu151Di':'H3K9me2', ' Sm152Di':'H4K16ac', ' Eu153Di':'H2AK119Ub', ' Gd155Di':'H3.3',
                   ' Gd156Di':'H3K64ac', ' Gd158Di':'ZEB1', ' Tb159Di':'H4', ' Gd160Di':'H3K27ac', ' Dy161Di':'H4K20me3', ' Ho165Di':'H3K36me3',
                   ' Er168Di':'H3K27me3', ' Tm169Di':'H3K9ac', ' Er170Di':'H3K9me3', ' Lu175Di':'H3S28p', ' Pr141Di':'human-EpCAM',
                    ' Sm147Di':'yH2A.X', ' Sm154Di':'Vimentin', ' Dy163Di':'ER', ' Dy164Di':'CD49f', ' Er166Di':'CD24', ' Er167Di':'GATA3',
                   ' Yb171Di':'CD44', ' Yb172Di':'Ki-67', ' Yb174Di':'K8_18'}


NamesAll_M=['H3.3', 'H4', 'H3K27Ac', 'H3K27me3', 'Irridium']
EpiCols_M=['H3.3', 'H4', 'H3K27Ac', 'H3K27me3']

In [ ]:
# file_object = open("CyTOF_Data/RepC1.csv", "w")
# for n in N_C1:
#     file_object.write(f'{n}\n')
# file_object.close()

In [ ]:
R=pd.read_csv("CyTOF_Data/RepC1.csv",header=None,)
R.columns=['C1','C2']
dictionary = dict(zip(R['C1'], R['C2']))

In [ ]:
dir="CyTOF_Data/"
import fcsparser

def LoadCyTOFData(CNum,Line,RepName):
    fName=f'CyTOF{CNum}_{Line}.fcs'
    meta, data = fcsparser.parse(dir+fName, reformat_meta=True)
    R=pd.read_csv(dir+RepName,header=None,)
    R.columns=['C1','C2']
    dictionary = dict(zip(R['C1'], R['C2']))
    data.rename(columns=dictionary,inplace=True)
    data=data[list(R['C2'])]
    DB=data.copy()
    return DB,meta.copy()

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

# Load From CyTOF1
C1_HCC70,_=LoadCyTOFData(1,'HCC70','RepC1.csv')
C1_HCC1937,_=LoadCyTOFData(1,'HCC1937','RepC1.csv')
C1_MDAMB468_Yael,_=LoadCyTOFData(1,'MDAMB468_Yael','RepC1.csv')
C1_MDAMB468_wo_CO2,_=LoadCyTOFData(1,'MDAMB468_wo_CO2','RepC1.csv')
C1_SUM149,_=LoadCyTOFData(1,'SUM149','RepC1.csv')
C1_MCF7_Ori,_=LoadCyTOFData(1,'MCF7_Ori','RepC1.csv')
C1_MCF7_Yael,_=LoadCyTOFData(1,'MCF7_Yael','RepC1.csv')

C5_DMSO,_=LoadCyTOFData(5,'DMSO','RepC5.csv')
C5_EZH2,_=LoadCyTOFData(5,'EZH2','RepC5.csv')

In [ ]:
C5_DMSO.shape

In [ ]:
NamesAll_C1=['Cytokeratin5',
 'H4K20me3',
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9me3',
 'H3K9me2',
 'H2AK119Ub',
 'H3.3',
 'H3K64ac',
 'ZEB1',
 'H3K27ac',
 'H3K36me3',
 'H3',
 'H3S28p',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'H3K4me1',
 'EpCAM',
 'yH2A.X',
 'H3K36me2',
 'H4K16ac',
 'Vimentin',
 'H4',
 'H3K9ac',
 'CD44',
 'Ki-67',
 'K8']

EpiCols_C1=[
 'H4K20me3',
 'H3K27me3',
 'H3K9me3',
 'H3K9me2',
 'H2AK119Ub',
 'H3.3',
 'H3K64ac',
 'H3K27ac',
 'H3K36me3',
 'H3',
 'H3S28p',
 'H3K27me2',
 'p53',
 'H3K4me3',
 'H3K4me1',
 'yH2A.X',
 'H3K36me2',
 'H4K16ac',
 'H4',
 'H3K9ac',
]

NormMRK_C1=[
 'Cytokeratin5',
 'ER',
 'EZH2',
 'GATA3',
 'H2A',
 'H3',
 'H3.3',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H3S28p',
 'H4',
 'H4K16ac',
 'H4K20me3',
 'K8',
 'Klf4',
 'Ki-67',
 'Vimentin',
 'ZEB1',
 'p53',
 'yH2A.X']

NormMRK_C1=list(set(NamesAll_C1).intersection(NormMRK))

NamesAll_C1.sort()
EpiCols_C1.sort()

In [ ]:
NamesAll_C5=['BMI-1',
 'CD24',
 'CD44',
 'CD49f',
 'CyclinB1',
 'Cytokeratin5',
 'DNA1',
 'DNA2',
 'ER',
 'EZH2',
 'EpCAM',
 'GATA3',
 'H2A',
 'H3',
 'H3.3',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H3S28p',
 'H4',
 'H4K16ac',
 'H4K20me3',
 'IdU',
 'K8',
 'Ki-67',
 'Klf4',
 'Vimentin',
 'ZEB1',
 'p53',
 'pRb',
 'yH2A.X']

EpiCols_C5=[
 'H2A',
 'H3',
 'H3.3',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H3S28p',
 'H4',
 'H4K16ac',
 'H4K20me3',
 'yH2A.X']



NonEpi=[
 'BMI-1',
 'CD24',
 'CD44',
 'CD49f',
 'CyclinB1',
 'Cytokeratin5',
 'ER',
 'EZH2',
 'EpCAM',
 'GATA3',
 'IdU',
 'K8',
 'Ki-67',
 'Klf4',
 'Vimentin',
 'ZEB1',
 'p53',
 'pRb',
]


NormMRK_C5=['BMI-1',
 'Cytokeratin5',
 'ER',
 'EZH2',
 'GATA3',
 'H2A',
 'H3',
 'H3.3',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H3S28p',
 'H4',
 'H4K16ac',
 'H4K20me3',
 'K8',
 'Klf4',
 'Ki-67',
 'Vimentin',
 'ZEB1',
 'p53',
 'yH2A.X']

# Gate on H3.3/H4 too low, but also remove outliers 99.99% from all 

In [ ]:
C5_DMSO.shape

In [ ]:
GateColumns=['H3.3','H4','H3']#,'H3']#,'H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data

C1_HCC70=Gate(C1_HCC70,"")
C1_HCC1937=Gate(C1_HCC1937,"")
C1_MDAMB468_Yael=Gate(C1_MDAMB468_Yael,"")
C1_MDAMB468_wo_CO2=Gate(C1_MDAMB468_wo_CO2,"")
C1_SUM149=Gate(C1_SUM149,"")
C1_MCF7_Ori=Gate(C1_MCF7_Ori,"")
C1_MCF7_Yael=Gate(C1_MCF7_Yael,"")
C5_DMSO=Gate(C5_DMSO,"")
C5_EZH2=Gate(C5_EZH2,"")


In [ ]:
NamesAll=list(set(NamesAll_C1).intersection(set(NamesAll_C5)))
EpiCols=list(set(EpiCols_C1).intersection(set(EpiCols_C5)))
NormMRK=list(set(NormMRK_C1).intersection(set(NormMRK_C5)))

In [ ]:
C5_DMSO.shape

In [ ]:
A,_=fitDF(C1_MCF7_Yael,C5_DMSO,C5_EZH2,MixedFit)
C5_EZH2.reset_index(inplace=True)
for C in NamesAll:
    if C in NamesAll:
        ms=C5_EZH2[C]>0 
    else:
        ms=[True]*len(C5_EZH2)
    C5_EZH2.loc[ms,C]=A.loc[ms,C]


A,_=fitDF(C1_MCF7_Yael,C5_DMSO,C5_DMSO,MixedFit)
C5_DMSO.reset_index(inplace=True)
for C in NamesAll:
    if C in NamesAll:
        ms=C5_DMSO[C]>0 
    else:
        ms=[True]*len(C5_DMSO)
    C5_DMSO.loc[ms,C]=A.loc[ms,C]


In [ ]:
NamesAll.sort()
EpiCols.sort()
NormMRK.sort()

C1_HCC70=C1_HCC70[NamesAll]
C1_HCC1937=C1_HCC1937[NamesAll]
C1_MDAMB468_Yael=C1_MDAMB468_Yael[NamesAll]
C1_MDAMB468_wo_CO2=C1_MDAMB468_wo_CO2[NamesAll]
C1_SUM149=C1_SUM149[NamesAll]
C1_MCF7_Ori=C1_MCF7_Ori[NamesAll]
C1_MCF7_Yael=C1_MCF7_Yael[NamesAll]
C5_DMSO=C5_DMSO[NamesAll]
C5_EZH2=C5_EZH2[NamesAll]

In [ ]:

scFac=5
C1_HCC70=np.arcsinh(C1_HCC70/scFac)
C1_HCC1937=np.arcsinh(C1_HCC1937/scFac)
C1_MDAMB468_Yael=np.arcsinh(C1_MDAMB468_Yael/scFac)
C1_MDAMB468_wo_CO2=np.arcsinh(C1_MDAMB468_wo_CO2/scFac)
C1_SUM149=np.arcsinh(C1_SUM149/scFac)
C1_MCF7_Ori=np.arcsinh(C1_MCF7_Ori/scFac)
C1_MCF7_Yael=np.arcsinh(C1_MCF7_Yael/scFac)
C5_DMSO=np.arcsinh(C5_DMSO/scFac)
C5_EZH2=np.arcsinh(C5_EZH2/scFac)


In [ ]:
a=[]
fig, axs = plt.subplots(10, 4, figsize=(15,30))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)
        
for i,N in enumerate(NamesAll):
    print(N)
    try:
        sns.histplot(data=C1_MCF7_Yael,x=N,ax=a[i],color='r',**hKWD)    
    except:
        pass
    try:
        sns.histplot(data=C5_DMSO,x=N,ax=a[i],color='b',**hKWD)    
    except:
        pass
    try:
#        pass
        sns.histplot(data=C5_EZH2,x=N,ax=a[i],color='g',**hKWD)    
    except:
        pass
#    a[i].set_title(N)

a[3].legend(bbox_to_anchor=(1,1),loc='upper left')
plt.subplots_adjust(wspace=0.75, hspace=0.7)
#fig.suptitle('MCF7 - Raw Data',y=0.91);
#fig.savefig("Plots/HCC1937-Raw.png",dpi=200,bbox_inches='tight')

In [ ]:
np.random.seed(42)

# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']**2+d.std()['H4']**2+d.std()['H3']**2

def NormalizeNew(data,MRK):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[MRK]=data[MRK]
    data=ddf2.copy()
    print(data.shape,ddf2.shape)
    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.3,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf, ddf,Q,M,M1,M2),method='cg')
    AA=out.params['a'].value

    M=M1*AA+M2*(1-AA)
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[NormMRK]=data[NormMRK]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:

# C1_HCC70=np.arcsinh(C1_HCC70/scFac)
# C1_HCC1937=np.arcsinh(C1_HCC1937/scFac)
# C1_MDAMB468_Yael=np.arcsinh(C1_MDAMB468_Yael/scFac)
# C1_MDAMB468_wo_CO2=np.arcsinh(C1_MDAMB468_wo_CO2/scFac)
# C1_SUM149=np.arcsinh(C1_SUM149/scFac)
# C1_MCF7_Ori=np.arcsinh(C1_MCF7_Ori/scFac)
# C1_MCF7_Yael=np.arcsinh(C1_MCF7_Yael/scFac)
# C5_DMSO=np.arcsinh(C5_DMSO/scFac)
# C5_EZH2=np.arcsinh(C5_EZH2/scFac)

C1_HCC70=NormalizeNew(C1_HCC70,NormMRK)
C1_HCC1937=NormalizeNew(C1_HCC1937,NormMRK)
C1_MDAMB468_Yael=NormalizeNew(C1_MDAMB468_Yael,NormMRK)
C1_MDAMB468_wo_CO2=NormalizeNew(C1_MDAMB468_wo_CO2,NormMRK)
C1_SUM149=NormalizeNew(C1_SUM149,NormMRK)
C1_MCF7_Ori=NormalizeNew(C1_MCF7_Ori,NormMRK)
C1_MCF7_Yael=NormalizeNew(C1_MCF7_Yael,NormMRK)
C5_DMSO=NormalizeNew(C5_DMSO,NormMRK)
C5_EZH2=NormalizeNew(C5_EZH2,NormMRK)



In [ ]:
from tqdm import tqdm



def MeanDev(DB,cols,MRK):
    DDB=DB.copy()
    Mean_Core=DDB[cols].mean(axis=1)
    for N in tqdm(MRK):
        DDB[N]=DDB[N]/Mean_Core
    return DDB

MeanMRK=['H3','H3.3','H4']
C1_HCC70=MeanDev(C1_HCC70,MeanMRK,NormMRK)
C1_HCC1937=MeanDev(C1_HCC1937,MeanMRK,NormMRK)
C1_MDAMB468_Yael=MeanDev(C1_MDAMB468_Yael,MeanMRK,NormMRK)
C1_MDAMB468_wo_CO2=MeanDev(C1_MDAMB468_wo_CO2,MeanMRK,NormMRK)
C1_SUM149=MeanDev(C1_SUM149,MeanMRK,NormMRK)
C1_MCF7_Ori=MeanDev(C1_MCF7_Ori,MeanMRK,NormMRK)
C1_MCF7_Yael=MeanDev(C1_MCF7_Yael,MeanMRK,NormMRK)
C5_DMSO=MeanDev(C5_DMSO,MeanMRK,NormMRK)
C5_EZH2=MeanDev(C5_EZH2,MeanMRK,NormMRK)

In [ ]:
C1_HCC70_B=C1_HCC70.copy()
C1_HCC1937_B=C1_HCC1937.copy()
C1_MDAMB468_Yael_B=C1_MDAMB468_Yael.copy()
C1_MDAMB468_wo_CO2_B=C1_MDAMB468_wo_CO2.copy()
C1_MCF7_Ori_B=C1_MCF7_Ori.copy()
C1_MCF7_Yael_B=C1_MCF7_Yael.copy()
C1_SUM149_B=C1_SUM149.copy()

C5_DMSO_B=C5_DMSO.copy()
C5_EZH2_B=C5_EZH2.copy()



aaa = pd.concat([
    C1_HCC70.sample(10000,replace=False,random_state=42),
    C1_HCC1937.sample(10000,replace=False,random_state=42),
    C1_MDAMB468_Yael.sample(10000,replace=False,random_state=42),
    C1_MDAMB468_wo_CO2.sample(10000,replace=False,random_state=42),
    C1_SUM149.sample(10000,replace=False,random_state=42),
    C1_MCF7_Ori.sample(10000,replace=False,random_state=42),
    C1_MCF7_Yael.sample(10000,replace=False,random_state=42),
]).copy()
                
m=np.mean(aaa,axis=0)
s=np.std(aaa,axis=0)

C1_HCC70=(C1_HCC70-m)/s
C1_HCC1937=(C1_HCC1937-m)/s
C1_MDAMB468_Yael=(C1_MDAMB468_Yael-m)/s
C1_MDAMB468_wo_CO2=(C1_MDAMB468_wo_CO2-m)/s
C1_SUM149=(C1_SUM149-m)/s
C1_MCF7_Ori=(C1_MCF7_Ori-m)/s
C1_MCF7_Yael=(C1_MCF7_Yael-m)/s
#C5_DMSO=(C5_DMSO-m)/s
#C5_EZH2=(C5_EZH2-m)/s


aaa = pd.concat([
    C5_DMSO.sample(10000,replace=False,random_state=42),
    C5_EZH2.sample(10000,replace=False,random_state=42),

]).copy()
                
m=np.mean(aaa,axis=0)
s=np.std(aaa,axis=0)


C5_DMSO=(C5_DMSO-m)/s
C5_EZH2=(C5_EZH2-m)/s

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

In [ ]:
C1_HCC70['Line']='HCC70'
C1_HCC1937['Line']='HCC1937'
C1_MDAMB468_Yael['Line']='MDAMB468 Yael'
C1_MDAMB468_wo_CO2['Line']='MDAMB468 CO2'
C1_SUM149['Line']='SUM149'
C1_MCF7_Ori['Line']='MCF7 Ori'
C1_MCF7_Yael['Line']='MCF7 Yael'
C5_DMSO['Line']='DMSO'
C5_EZH2['Line']='EZH2'

Colors={
 'HCC70': 'r',
 'HCC1937': 'g',
 'MDAMB468 Yael': 'b',
 'MDAMB468 CO2': 'mediumseagreen',
 'SUM149': 'magenta',
 'MCF7 Ori': 'orange',
 'MCF7 Yael': 'thistle',
 'DMSO': 'gold',
 'EZH2': 'aqua'}


Colors = {
    'HCC70': '#FF6347',  # Tomato
    'HCC1937': '#90EE90',  # Light green
    'MDAMB468 Yael': '#1E90FF',  # Dodger blue
    'MDAMB468 CO2': '#00BFFF',  # Deep sky blue
    'SUM149': '#FF00FF',  # Fuchsia
    'MCF7 Ori': '#FFA500',  # Orange
    'MCF7 Yael': '#FFD700',  # Gold
    'DMSO': '#DAA520',  # Golden rod
    'EZH2': '#7FFFD4'  # Aquamarine
}


In [ ]:
C1_HCC70=C1_HCC70_B.copy()
C1_HCC1937=C1_HCC1937_B.copy()
C1_MDAMB468_Yael=C1_MDAMB468_Yael_B.copy()
C1_MDAMB468_wo_CO2=C1_MDAMB468_wo_CO2_B.copy()
C1_MCF7_Ori=C1_MCF7_Ori_B.copy()
C1_MCF7_Yael=C1_MCF7_Yael_B.copy()
C1_SUM149=C1_SUM149_B.copy()

C5_DMSO=C5_DMSO_B.copy()
C5_EZH2=C5_EZH2_B.copy()


C5_DMSO=C5_DMSO[NamesAll]
C5_EZH2=C5_EZH2[NamesAll]


aaa = pd.concat([
    C1_HCC70.sample(10000,replace=False),
    C1_HCC1937.sample(10000,replace=False),
    C1_MDAMB468_Yael.sample(10000,replace=False),
    C1_MDAMB468_wo_CO2.sample(10000,replace=False),
    C1_SUM149.sample(10000,replace=False),
    C1_MCF7_Ori.sample(10000,replace=False),
    C1_MCF7_Yael.sample(10000,replace=False),
    C5_DMSO.sample(10000,replace=False),
    C5_EZH2.sample(10000,replace=False),
]).copy()
                
m=np.mean(aaa,axis=0)
s=np.std(aaa,axis=0)

C1_HCC70=(C1_HCC70-m)/s
C1_HCC1937=(C1_HCC1937-m)/s
C1_MDAMB468_Yael=(C1_MDAMB468_Yael-m)/s
C1_MDAMB468_wo_CO2=(C1_MDAMB468_wo_CO2-m)/s
C1_SUM149=(C1_SUM149-m)/s
C1_MCF7_Ori=(C1_MCF7_Ori-m)/s
C1_MCF7_Yael=(C1_MCF7_Yael-m)/s
C5_DMSO=(C5_DMSO-m)/s
C5_EZH2=(C5_EZH2-m)/s




In [ ]:
C1_HCC70['Line']='HCC70'
C1_HCC1937['Line']='HCC1937'
C1_MDAMB468_Yael['Line']='MDAMB468 Yael'
C1_MDAMB468_wo_CO2['Line']='MDAMB468 CO2'
C1_SUM149['Line']='SUM149'
C1_MCF7_Ori['Line']='MCF7 Ori'
C1_MCF7_Yael['Line']='MCF7 Yael'
C5_DMSO['Line']='DMSO'
C5_EZH2['Line']='EZH2'

In [ ]:
CyTOF1_MCF7_Cl=np.loadtxt("Data/CyTOF5_MCF7_CellID_Labels.csv")
CyTOF1_EZH2i_Cl=np.loadtxt("Data/CyTOF5_EZH2i_CellID_Labels.csv")

In [ ]:
np.unique(CyTOF1_MCF7_Cl)

In [ ]:
CNum=15000
CAll=pd.concat([
    C1_HCC1937.sample(CNum,replace=False),
    C1_MDAMB468_Yael.sample(CNum,replace=False),
#    C1_MCF7_Yael.sample(CNum,replace=False),
]).copy()

In [ ]:
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
DefStyle()
for NN in ['Cytokeratin5']:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("(CellIden) "+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');

In [ ]:
labels_1=dbscan_plot(X_2d,eps=0.11
                   ,min_samples=100,)

In [ ]:
labels_1[labels_1!=0]=20
labels_1[labels_1==0]=1
labels_1[labels_1==20]=0

In [ ]:
import xgboost as xgb
XGC_SuperBasal=xgb.XGBClassifier().fit(CAll[EPC],labels_1)

In [ ]:
np.unique(labels_1, return_counts=True)

In [ ]:
a=[]
fig, axs = plt.subplots(5, 2, figsize=(10,15))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)

for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
#    print(i,L)
    M=CAll.Line==L
    
    a[i].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',
                )

    a[i].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L])

M=labels_1==1
a[2].scatter(X_2d[:,0],X_2d[:,1],s=2, color='gray')
a[2].scatter(X_2d[M,0],X_2d[M,1],s=2, color='r')
 
for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
    M=CAll.Line==L
    a[3].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L])

a[3].legend(markerscale=10,loc='upper right',fontsize=10,bbox_to_anchor=(1.5,1))



In [ ]:
C1_HCC1937['SuperBasal']=XGC_SuperBasal.predict(C1_HCC1937[EPC])
C1_MDAMB468_Yael['SuperBasal']=XGC_SuperBasal.predict(C1_MDAMB468_Yael[EPC])

In [ ]:
CAll=C1_HCC1937.sample(CNum,replace=False,random_state=42).copy()
X_2d=draw_umap(CAll[CellIden],
               cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
DefStyle()
for NN in ['Cytokeratin5']:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("(CellIden) "+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');

In [ ]:
CAll=C1_MDAMB468_Yael.sample(CNum,replace=False,random_state=42).copy()
X_2d=draw_umap(CAll[CellIden],
               cc=CAll['Cytokeratin5'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()


In [ ]:
DefStyle()
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("(CellIden) "+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');
    plt.savefig("Plots/CyTOF1_MDAMB468_CellIden_"+NN+".png",dpi=200,bbox_inches='tight')

In [ ]:
#C1_MCF7_Yael['Cl']=Cl_Yael
#C1_MCF7_Ori['Cl']=Cl_Ori
C5_DMSO['OldCl']=CyTOF1_MCF7_Cl
C5_EZH2['OldCl']=CyTOF1_EZH2i_Cl

In [ ]:
CNum=15000
CAll=pd.concat([
#    C1_HCC70.sample(CNum,replace=False),
    C1_HCC1937.sample(CNum,replace=False),
    C1_MDAMB468_Yael.sample(CNum,replace=False),
#    C1_MDAMB468_wo_CO2.sample(CNum,replace=False),
#    C1_SUM149.sample(CNum,replace=False),
#    C1_MCF7_Ori.sample(CNum,replace=False),
    C1_MCF7_Yael.sample(CNum,replace=False),

    C5_DMSO.sample(CNum,replace=False),
    C5_EZH2.sample(CNum,replace=False),
]).copy()

In [ ]:
M=CAll.OldCl.isna()

In [ ]:
CAll.loc[M,'OldCl']=999

In [ ]:
M=CAll.SuperBasal.isna()
CAll.loc[M,'SuperBasal']=0

In [ ]:
CAll.groupby('SuperBasal').count()

In [ ]:
X_2d=draw_umap(CAll[CellIden],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=None)
plt.show()

In [ ]:
#plt.figure(figsize=(5,5))

a=[]
fig, axs = plt.subplots(6, 2, figsize=(10,15))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)

for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
#    print(i,L)
    M=CAll.Line==L
    
    a[i].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',
                )

    a[i].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L]
                )
    # M=CAll.Cl==2
    # a[i].scatter(X_2d[M,0],X_2d[M,1],s=2,c='darkslategray',)
    # M=CAll.Cl==3
    # a[i].scatter(X_2d[M,0],X_2d[M,1],s=2,c='khaki',)
    # a[i].set_ylim(X_2d[:,1].min()*1.1,X_2d[:,1].max()*1.1)
    # a[i].set_xlim(X_2d[:,0].min()*0.9,X_2d[:,0].max()*1.1)
#    a[i].legend(markerscale=10,loc='upper right',fontsize=10)


M=CAll.OldCl==1
a[5].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',)
a[5].scatter(X_2d[M,0],X_2d[M,1],s=2,c='black',label='CyTOF5 EZH2 Main')
a[5].legend(markerscale=10,bbox_to_anchor=(1,1),fontsize=10,)

M=CAll.OldCl==2
a[6].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',)
a[6].scatter(X_2d[M,0],X_2d[M,1],s=2,c='magenta',label='CyTOF5 Basal Like')
a[6].legend(markerscale=10,bbox_to_anchor=(-.1,1),fontsize=10,)

M=CAll.OldCl==0
a[7].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',)
a[7].scatter(X_2d[M,0],X_2d[M,1],s=2,c='red',label='CyTOF5 Luminal')
a[7].legend(markerscale=10,bbox_to_anchor=(1,1),fontsize=10,)

M=CAll.SuperBasal==1
a[8].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',)
a[8].scatter(X_2d[M,0],X_2d[M,1],s=2,c='green',label='CyTOF1 SuperBasal')
a[8].legend(markerscale=10,bbox_to_anchor=(-.1,1),fontsize=10,)

for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
    M=CAll.Line==L
    


    a[11].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L]
                )


a[11].legend(markerscale=10,loc='upper right',fontsize=10,bbox_to_anchor=(1.5,1))

fig.savefig("Plots/CyTOF1_5_Joint_CellIden.png",dpi=200,bbox_inches='tight')

In [ ]:
DefStyle()
for NN in ['Cytokeratin5']:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))

    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("(CellIden) "+TSNEVar)
    plt.xlabel('UMAP 0');
    plt.ylabel('UMAP 1');


    plt.savefig('Plots/Joined_CyTOF1_5_CellIden_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
#plt.figure(figsize=(5,5))

a=[]
fig, axs = plt.subplots(5, 2, figsize=(10,20))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)

for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
#    print(i,L)
    M=CAll.Line==L
    
    a[i].scatter(X_2d[:,0],X_2d[:,1],s=2, c='gray',
                )

    a[i].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L]
                )
    M=CAll.Cl==1
#    a[i].scatter(X_2d[M,0],X_2d[M,1],s=2,c='black',alpha=1)
    M=CAll.Cl==2
#    a[i].scatter(X_2d[M,0],X_2d[M,1],s=2,c='darkslategray',alpha=01)
    # a[i].set_ylim(X_2d[:,1].min()*1.1,X_2d[:,1].max()*1.1)
    # a[i].set_xlim(X_2d[:,0].min()*0.9,X_2d[:,0].max()*1.1)
#    a[i].legend(markerscale=10,loc='upper right',fontsize=10)

for i,L in enumerate(CAll.Line.unique()):
    #a[1].figure(figsize=(5,5))
#    print(i,L)
    M=CAll.Line==L
    


    a[9].scatter(X_2d[M,0],X_2d[M,1],s=2, label=L,c=Colors[L]
                )

    # a[9].set_ylim(X_2d[:,1].min()*1.1,X_2d[:,1].max()*1.1)
    # a[9].set_xlim(X_2d[:,0].min()*0.9,X_2d[:,0].max()*1.1)

M=CAll.Cl==1
a[9].scatter(X_2d[M,0],X_2d[M,1],s=2, label='Basal Like CyTOF 1',c='black',)
M=CAll.Cl==2
a[9].scatter(X_2d[M,0],X_2d[M,1],s=2, label='Basal Like CyTOF 5',c='darkslategray',)
a[9].legend(markerscale=10,loc='upper right',fontsize=10,bbox_to_anchor=(1.5,1))

fig.savefig("Plots/CyTOF1_5_Joint_HM.png",dpi=200,bbox_inches='tight')

In [ ]:
sns.heatmap(C1_MCF7_Yael.groupby('Cl').mean(numeric_only=True)[CellIden].T,annot=True,cmap=plt.cm.seismic,vmin=-1,vmax=1)

In [ ]:
sns.heatmap(C5_DMSO.groupby('Cl').mean(numeric_only=True)[CellIden].T,annot=True,cmap=plt.cm.seismic,vmin=-1,vmax=1)

In [ ]:
Mat=CAll.groupby(['Line','Cl']).mean(numeric_only=True)

In [ ]:
CAll['SubL']=CAll['Line']
M=CAll.Cl==1
CAll.loc[M,'SubL']='CyTOF 1 Basal Like'
M=CAll.Cl==2
CAll.loc[M,'SubL']='CyTOF 5 Basal Like'
M=CAll.Cl==3
CAll.loc[M,'SubL']='EZH2i Main'

In [ ]:
Mat=CAll.groupby('SubL').mean(numeric_only=True)

In [ ]:
plt.figure(figsize=(10,10))
sns.clustermap(Mat[CellIden].T,annot_kws={'size':8},annot=True,cmap=plt.cm.seismic,vmin=-1.5,vmax=1.5,row_cluster=False)
plt.savefig("Plots/CyTOF1_5_HeatMap_CellIden.png",dpi=200,bbox_inches='tight')

In [ ]:
plt.figure(figsize=(10,10))
sns.clustermap(Mat[EPC].T,annot_kws={'size':8},annot=True,cmap=plt.cm.seismic,vmin=-1.5,vmax=1.5,row_cluster=False)
plt.savefig("Plots/CyTOF1_5_HeatMap_Epi.png",dpi=200,bbox_inches='tight')